# R21 Covariate Randomise Scatterplots

Inspect the covariate-adjusted randomise results for C3/C4 only. C3 and C4 test the final covariate column in each model, so they correspond to blink or pupil effects after including mean FD as the adjustment covariate. C1/C2 mean-effect rows are intentionally ignored here.

## 1. Verify notebook dependencies

Launch this notebook with `bash notebooks/run_covariate_randomise_notebook.sh`. It only needs the tracked covariate randomise summary files, not the large Linux-only dual-regression folders.

In [ ]:
from pathlib import Path
import importlib.util

def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'code' / 'check_covariate_randomise_results.py').is_file():
            return candidate
    raise FileNotFoundError('Start Jupyter from the r21-rest repository or one of its subdirectories.')

PROJECT_ROOT = find_project_root()
required_imports = ('matplotlib', 'numpy', 'pandas', 'scipy')
missing = [name for name in required_imports if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError('Missing packages: ' + ', '.join(missing) + '. Relaunch with notebooks/run_covariate_randomise_notebook.sh')
print(f'Project root: {PROJECT_ROOT}')

## 2. Load C3/C4 covariate rows

FSL corrected-p images store `1-p`, so `peak_corrp > 0.95` corresponds to corrected `p < 0.05`. These rows are the covariate-effect tests only.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display, Markdown

CORRP_THRESHOLD = 0.95
SUMMARY = PROJECT_ROOT / 'derivatives' / 'fsl' / 'covariate_randomise_summary' / 'task-rest_covariate-randomise_peak_summary.tsv'
if not SUMMARY.is_file():
    raise FileNotFoundError(SUMMARY)

summary = pd.read_csv(SUMMARY, sep='\t', dtype=str, keep_default_na=False)
summary['peak_corrp_num'] = pd.to_numeric(summary['peak_corrp'], errors='coerce')
summary['corrected_p'] = 1.0 - summary['peak_corrp_num']
summary['has_roi_values'] = summary['roi_values_tsv'].str.strip().ne('')

covariate_rows = summary[
    summary['design_contrast'].isin(['C3', 'C4'])
    & summary['inference'].eq('cluster-extent')
].copy()
covariate_rows['effect_label'] = (
    covariate_rows['model_label'].str.replace('cov-fdmean-', '', regex=False).str.title()
    + ' '
    + covariate_rows['design_contrast']
    + ' '
    + covariate_rows['contrast_name']
)

n_significant = int((covariate_rows['peak_corrp_num'] > CORRP_THRESHOLD).sum())
display(Markdown(
    f'Loaded **{len(summary)}** total covariate-randomise rows and **{len(covariate_rows)}** C3/C4 rows. '
    f'Completed C3/C4 rows above `1-p > {CORRP_THRESHOLD}`: **{n_significant}**.'
))

audit_columns = [
    'peak_corrp', 'corrected_p', 'model_label', 'analysis', 'network', 'component',
    'condition_contrast', 'design_contrast', 'contrast_name', 'tested_covariate',
    'expected_participants', 'status', 'roi_values_tsv'
]
display(
    covariate_rows.sort_values('peak_corrp_num', ascending=False)[audit_columns]
    .style.format({'corrected_p': '{:.4f}'})
)

## 3. C3/C4 corrected-p audit

This plot summarizes the strongest covariate-effect maps. It is not a scatterplot; it is a quick audit of whether any C3/C4 map crossed the corrected threshold and which models came closest.

In [ ]:
top_n = min(18, len(covariate_rows))
top = covariate_rows.sort_values('peak_corrp_num', ascending=False).head(top_n).iloc[::-1].copy()
labels = (
    top['model_label'].str.replace('cov-fdmean-', '', regex=False)
    + ' | ' + top['analysis']
    + ' | ' + top['network']
    + ' c' + top['component']
    + ' | ' + top['condition_contrast']
    + ' | ' + top['design_contrast']
)
colors = np.where(top['peak_corrp_num'] > CORRP_THRESHOLD, '#0072B2', '#8d99ae')

fig_height = max(5, 0.38 * top_n)
fig, ax = plt.subplots(figsize=(11, fig_height))
ax.barh(labels, top['peak_corrp_num'], color=colors)
ax.axvline(CORRP_THRESHOLD, color='#d62828', linestyle='--', linewidth=1.5, label='corrected p = .05')
ax.set_xlim(0, 1)
ax.set_xlabel('Peak corrected map value (1 - p)')
ax.set_title('Top C3/C4 covariate-effect randomise peaks')
ax.legend(loc='lower right')
for spine in ('top', 'right'):
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.show()

## 4. C3/C4 scatterplots

Scatterplots use tracked `*_subjectContrast_values.tsv` files. For each significant C3/C4 map, the x axis is the blink or pupil contrast delta and the y axis is the subject-level brain contrast beta extracted from the corrected map ROI. Mean FD remains in the model but is not plotted on the x axis. If no C3/C4 ROI tables exist, the cell reports that explicitly.

In [ ]:
COVARIATE_LABELS = {
    'delta_blink_rate_per_min': 'Blink-rate contrast delta (per min)',
    'delta_mean_pupil_area': 'Mean pupil-area contrast delta',
}

def project_path(value):
    path = Path(value)
    return path if path.is_absolute() else PROJECT_ROOT / path

def describe_row(row):
    model = row['model_label'].replace('cov-fdmean-', '').title()
    return (
        f"{model}: {row['analysis']} {row['network']} comp {row['component']} | "
        f"{row['condition_contrast']} | {row['design_contrast']} {row['contrast_name']}"
    )

def plot_covariate_scatter(row):
    values_path = project_path(row['roi_values_tsv'])
    values = pd.read_csv(values_path, sep='\t', dtype=str, keep_default_na=False)
    x_col = row['tested_covariate']
    y_col = 'subject_contrast_beta'
    if x_col not in values:
        raise KeyError(f'{x_col} missing from {values_path}')
    values[x_col] = pd.to_numeric(values[x_col], errors='coerce')
    values[y_col] = pd.to_numeric(values[y_col], errors='coerce')
    values = values.dropna(subset=[x_col, y_col]).copy()
    if len(values) < 3:
        raise ValueError(f'Not enough complete rows for {values_path}')

    pearson = stats.pearsonr(values[x_col], values[y_col])
    spearman = stats.spearmanr(values[x_col], values[y_col])
    slope, intercept, r_value, p_value, stderr = stats.linregress(values[x_col], values[y_col])
    x_line = np.linspace(values[x_col].min(), values[x_col].max(), 100)
    y_line = intercept + slope * x_line

    fig, ax = plt.subplots(figsize=(6.8, 5.2))
    ax.scatter(values[x_col], values[y_col], s=42, color='#0072B2', alpha=0.82, edgecolor='white', linewidth=0.6)
    ax.plot(x_line, y_line, color='#d62828', linewidth=2)
    ax.axhline(0, color='0.75', linewidth=1)
    ax.axvline(0, color='0.75', linewidth=1)
    ax.set_xlabel(COVARIATE_LABELS.get(x_col, x_col))
    ax.set_ylabel('Brain contrast beta in corrected ROI')
    ax.set_title(describe_row(row), fontsize=10)
    ax.text(
        0.02, 0.98,
        f"n = {len(values)}\nPearson r = {pearson.statistic:.2f}, p = {pearson.pvalue:.3g}\n"
        f"Spearman rho = {spearman.statistic:.2f}, p = {spearman.pvalue:.3g}",
        transform=ax.transAxes,
        va='top',
        ha='left',
        bbox={'boxstyle': 'round,pad=0.3', 'facecolor': 'white', 'edgecolor': '0.82'},
    )
    for spine in ('top', 'right'):
        ax.spines[spine].set_visible(False)
    plt.tight_layout()
    plt.show()
    return values

scatter_rows = covariate_rows[covariate_rows['roi_values_tsv'].str.strip().ne('')].copy()
if scatter_rows.empty:
    display(Markdown(
        '**No C3/C4 scatterplots are currently available.** The covariate randomise compiler only wrote ROI-value TSVs for maps with `1-p > 0.95`, and no C3/C4 maps crossed that corrected threshold in this run.'
    ))
    display(Markdown('Closest C3/C4 peaks are shown below for review.'))
    display(covariate_rows.sort_values('peak_corrp_num', ascending=False).head(12)[audit_columns].style.format({'corrected_p': '{:.4f}'}))
else:
    for _, row in scatter_rows.sort_values('peak_corrp_num', ascending=False).iterrows():
        display(Markdown(f"### {describe_row(row)}"))
        plot_covariate_scatter(row)